# Full-run test for ForestFlow

This notebook provides runnable cells to perform a full training run for 'ForestFlow' using the project `TrainTestSplitPipeline`.

Notes:
- ForestFlow uses XGBoost-based flow matching for fast tabular data generation
- No GPU required - runs efficiently on CPU with parallel processing
- Expected runtime: 10-25 minutes per dataset (32K rows)
- Uses pretrained XGBoost models with flow-based diffusion

In [1]:
# Install dependencies
%pip install ForestDiffusion xgboost scikit-learn

In [2]:
# Imports and helpers
import os
import importlib
import traceback
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess

def ensure(path):
    os.makedirs(path, exist_ok=True)

def make_pipeline(model_callable, skip_evaluations=True, evaluations=None):
    if skip_evaluations:
        return TrainTestSplitPipeline(model=model_callable, evaluations=[], override_evaluations=True)
    elif evaluations is not None:
        return TrainTestSplitPipeline(model=model_callable, evaluations=evaluations, override_evaluations=True)
    else:
        return TrainTestSplitPipeline(model=model_callable)

In [3]:
# Configuration
DATASETS = ['adult', 'car', 'magic', 'nursery', 'shuttle']
SKIP_EVALUATIONS = False  # Set to True to skip TSTR evaluations

# Top-level dirs
ensure('discretized_data')
ensure('sample_data')
ensure('synthetic')
ensure('Results')

MODEL_MAP = {
    'forestflow': ('katabatic.models.forestflow.models', 'ForestFlowModel'),
}

# Optimized config for faster runtime (10-15 min for 32K rows)
FORESTFLOW_CONFIG = {
    'n_t': 30,              # Reduced from default 50 for speed
    'duplicate_K': 75,      # Reduced from default 100 for speed
    'n_batch': 1,           # Use data iterator for memory efficiency
    'diffusion_type': 'flow',  # Flow-matching (faster than 'vp')
    'n_jobs': -1,           # Use all CPU cores
    'max_depth': 7,
    'n_estimators': 100,
    'random_state': 42,
}

# For even faster results (5-8 min), use this config:
# FORESTFLOW_CONFIG = {
#     'n_t': 20,
#     'duplicate_K': 50,
#     'n_batch': 1,
#     'diffusion_type': 'flow',
#     'n_jobs': -1,
#     'random_state': 42,
# }

## Preprocess datasets (run once)

In [4]:
for dataset in DATASETS:
    print('\n' + '='*60)
    print(f'Preprocessing {dataset}...')
    try:
        discretize_preprocess(
            file_path=f'raw_data/{dataset}.csv', 
            output_path=f'discretized_data/{dataset}.csv', 
            bins=10, 
            strategy='uniform'
        )
        print(f'Discretized -> discretized_data/{dataset}.csv')
    except Exception as e:
        print(f'Failed to preprocess {dataset}: {e}')
        traceback.print_exc()


Preprocessing adult...
Preprocessing: raw_data/adult.csv
Saved preprocessed discrete dataset to: discretized_data/adult.csv
Discretized -> discretized_data/adult.csv

Preprocessing car...
Preprocessing: raw_data/car.csv
Saved preprocessed discrete dataset to: discretized_data/car.csv
Discretized -> discretized_data/car.csv

Preprocessing magic...
Preprocessing: raw_data/magic.csv
Saved preprocessed discrete dataset to: discretized_data/magic.csv
Discretized -> discretized_data/magic.csv

Preprocessing nursery...
Preprocessing: raw_data/nursery.csv
Saved preprocessed discrete dataset to: discretized_data/nursery.csv
Discretized -> discretized_data/nursery.csv

Preprocessing shuttle...
Preprocessing: raw_data/shuttle.csv
Saved preprocessed discrete dataset to: discretized_data/shuttle.csv
Discretized -> discretized_data/shuttle.csv


## Run ForestFlow

In [ ]:
for dataset in DATASETS:
    print('\n' + '='*60)
    print(f'ForestFlow -> {dataset}')
    synth_dir = os.path.join('synthetic', dataset, 'forestflow')
    ensure(synth_dir)
    try:
        mod_path, cls_name = MODEL_MAP['forestflow']
        module = importlib.import_module(mod_path)
        ModelClass = getattr(module, cls_name)
        model_factory = lambda: ModelClass(**FORESTFLOW_CONFIG)
        pipeline = make_pipeline(model_factory, skip_evaluations=SKIP_EVALUATIONS)
        result = pipeline.run(
            input_csv=f'discretized_data/{dataset}.csv',
            output_dir=f'sample_data/{dataset}',
            synthetic_dir=synth_dir,
            real_test_dir=f'sample_data/{dataset}'
        )
        print('ForestFlow finished for', dataset)
    except Exception as e:
        print('ForestFlow failed for', dataset, e)
        traceback.print_exc()


ForestFlow -> adult
Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
0    0.759175
1    0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.759251
1    0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
[ForestFlow] Initializing with n_t=30, duplicate_K=75...
[ForestFlow] Using diffusion_type='flow' with n_batch=1
[ForestFlow] Detected 1 categorical, 12 integer, 1 binary columns
ForestFlow failed for adult bad allocation

ForestFlow -> car


joblib.externals.loky.process_executor._RemoteTraceback: 
"""
Traceback (most recent call last):
  File "C:\Users\lbrum\anaconda3\Lib\site-packages\joblib\externals\loky\process_executor.py", line 463, in _process_worker
    r = call_item()
        ^^^^^^^^^^^
  File "C:\Users\lbrum\anaconda3\Lib\site-packages\joblib\externals\loky\process_executor.py", line 291, in __call__
    return self.fn(*self.args, **self.kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lbrum\anaconda3\Lib\site-packages\joblib\parallel.py", line 598, in __call__
    return [func(*args, **kwargs)
            ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lbrum\anaconda3\Lib\site-packages\ForestDiffusion\diffusion_with_trees_class.py", line 246, in train_iterator
    out = xgb.train(xgb_dict, data_iterator, num_boost_round=self.n_estimators)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\lbrum\anaconda3\Lib\site-packages\xgboost\core.py", line 729, in

Loaded data with shape: (1728, 7)
Saved train/test full data
Train size: (1382, 7), Test size: (346, 7)
Train label distribution:
 6
2    0.700434
0    0.222142
1    0.039797
3    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
2    0.699422
0    0.222543
1    0.040462
3    0.037572
Name: proportion, dtype: float64
Saved X/y split
Training shape: (1382, 6) (1382,)
Test shape: (346, 6) (346,)
[ForestFlow] Initializing with n_t=30, duplicate_K=75...
[ForestFlow] Using diffusion_type='flow' with n_batch=1
[ForestFlow] Detected 0 categorical, 6 integer, 0 binary columns
